In [9]:
# 2.1 理论计算题 
# ============================================================================
from collections import Counter, defaultdict

# 字符序列
sequence = "ababc"
vocab = ['a', 'b', 'c']
vocab_size = len(vocab)

# 统计转移次数
transitions = defaultdict(Counter)
for i in range(len(sequence) - 1):
    current = sequence[i]
    next_char = sequence[i + 1]
    transitions[current][next_char] += 1
print("转移次数统计:")
for current in vocab:
    for next_char in vocab:
        count = transitions[current][next_char]
        if count > 0:
            print(f"  {current}→{next_char}: {count}次")

print("\n拉普拉斯平滑估计:")
# 计算条件概率 p(next|current) = (count(current→next) + 1) / (count(current→*) + |V|)
for current in vocab:
    total_transitions = sum(transitions[current].values())
    print(f"\n给定前驱词 '{current}':")
    print(f"  总转移次数: {total_transitions}")
    print(f"  词汇表大小 |V| = {vocab_size}")
    for next_char in vocab:
        count = transitions[current][next_char]
        prob = (count + 1) / (total_transitions + vocab_size)
        print(f"  p({next_char}|{current}) = ({count} + 1) / ({total_transitions} + {vocab_size}) = {prob:.3f}")

# 特别回答问题
print("\n" + "="*50)
print("问题答案:")
count_b_to_a = transitions['b']['a']
count_b_to_c = transitions['b']['c']
total_b = sum(transitions['b'].values())

p_a_given_b = (count_b_to_a + 1) / (total_b + vocab_size)
p_c_given_b = (count_b_to_c + 1) / (total_b + vocab_size)

print(f"1. p(a|b) = ({count_b_to_a} + 1) / ({total_b} + {vocab_size}) = {p_a_given_b}")
print(f"2. p(c|b) = ({count_b_to_c} + 1) / ({total_b} + {vocab_size}) = {p_c_given_b}")


转移次数统计:
  a→b: 2次
  b→a: 1次
  b→c: 1次

拉普拉斯平滑估计:

给定前驱词 'a':
  总转移次数: 2
  词汇表大小 |V| = 3
  p(a|a) = (0 + 1) / (2 + 3) = 0.200
  p(b|a) = (2 + 1) / (2 + 3) = 0.600
  p(c|a) = (0 + 1) / (2 + 3) = 0.200

给定前驱词 'b':
  总转移次数: 2
  词汇表大小 |V| = 3
  p(a|b) = (1 + 1) / (2 + 3) = 0.400
  p(b|b) = (0 + 1) / (2 + 3) = 0.200
  p(c|b) = (1 + 1) / (2 + 3) = 0.400

给定前驱词 'c':
  总转移次数: 0
  词汇表大小 |V| = 3
  p(a|c) = (0 + 1) / (0 + 3) = 0.333
  p(b|c) = (0 + 1) / (0 + 3) = 0.333
  p(c|c) = (0 + 1) / (0 + 3) = 0.333

问题答案:
1. p(a|b) = (1 + 1) / (2 + 3) = 0.4
2. p(c|b) = (1 + 1) / (2 + 3) = 0.4


In [8]:
# 2.2 编程题
# ============================================================================
import re
from collections import Counter

def preprocess_text(text, n):
    """
    预处理文本，构建词汇表，生成特征序列和标签序列
    
    Args:
        text: 输入文本
        n: 滑动窗口大小（特征序列长度）
    
    Returns:
        vocab: 词汇表字典 {词: ID}
        features: 特征列表
        labels: 标签列表
    """
    # 1. 转换为小写，去除标点符号（保留字母和空格）
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # 2. 按空格分词
    tokens = text.split()
    
    # 3. 构建词汇表（按出现频率排序，分配整数ID，从0开始）
    word_counts = Counter(tokens)
    # 按频率降序排序，频率相同按字母顺序
    sorted_words = sorted(word_counts.items(), key=lambda x: (-x[1], x[0]))
    vocab = {word: idx for idx, (word, _) in enumerate(sorted_words)}
    
    # 4. 用滑动窗口生成长度为n的特征序列和对应的下一个词标签
    features = []
    labels = []
    
    for i in range(len(tokens) - n):
        feature = tokens[i:i+n]
        label = tokens[i+n]
        features.append(feature)
        labels.append(label)
    
    return vocab, (features, labels)

# 测试
text = "The time machine"
n = 2
vocab, (features, labels) = preprocess_text(text, n)

print("\n" + "="*50)
print("2.2 测试结果:")
print("词汇表:", vocab)
print("特征:", features)
print("标签:", labels)



2.2 测试结果:
词汇表: {'machine': 0, 'the': 1, 'time': 2}
特征: [['the', 'time']]
标签: ['machine']


In [10]:
# 3.1 理论计算题 - 代码演示梯度计算
# ============================================================================
import numpy as np

def demonstrate_rnn_gradient():
    """
    演示线性RNN的梯度计算，验证梯度消失/爆炸
    """
    np.random.seed(42)
    
    # 设置参数
    T = 5  # 时间步长
    hidden_size = 2
    input_size = 3
    
    # 随机初始化权重
    W_hh = np.random.randn(hidden_size, hidden_size) * 0.5
    W_xh = np.random.randn(hidden_size, input_size) * 0.5
    W_oh = np.random.randn(input_size, hidden_size) * 0.5  # 为了演示，输出维度设为input_size
    
    # 生成随机输入和目标
    X = np.random.randn(T, input_size)
    y = np.random.randn(T, input_size)
    
    # 前向传播存储所有状态
    h = np.zeros((T + 1, hidden_size))
    o = np.zeros((T, input_size))
    
    for t in range(T):
        h[t+1] = np.dot(W_hh, h[t]) + np.dot(W_xh, X[t])
        o[t] = np.dot(W_oh, h[t+1])
    
    # 计算梯度（通过BPTT）
    # 先计算损失对输出的梯度
    dL_do = o - y  # (T, input_size)
    
    # 反向传播
    dh_next = np.zeros(hidden_size)
    dW_hh = np.zeros_like(W_hh)
    dW_xh = np.zeros_like(W_xh)
    dW_oh = np.zeros_like(W_oh)
    
    for t in range(T-1, -1, -1):
        # 损失对h_t的梯度
        dh_t = np.dot(W_oh.T, dL_do[t]) + dh_next
        
        # 累积梯度
        dW_hh += np.outer(dh_t, h[t])
        dW_xh += np.outer(dh_t, X[t])
        dW_oh += np.outer(dL_do[t], h[t+1])
        
        # 传递给上一时间步
        dh_next = np.dot(W_hh.T, dh_t)
    
    print("\n" + "="*50)
    print("3.1 RNN梯度演示:")
    print(f"W_hh的梯度范数: {np.linalg.norm(dW_hh):.6f}")
    print(f"W_hh的特征值: {np.linalg.eigvals(W_hh)}")
    print("如果特征值绝对值大于1 → 梯度爆炸")
    print("如果特征值绝对值小于1 → 梯度消失")
    
    return dW_hh, W_hh

dW_hh, W_hh = demonstrate_rnn_gradient()


3.1 RNN梯度演示:
W_hh的梯度范数: 16.565133
W_hh的特征值: [0.29650209 0.71336991]
如果特征值绝对值大于1 → 梯度爆炸
如果特征值绝对值小于1 → 梯度消失


In [11]:
# 3.2 编程题
# ============================================================================

def rnn_cell_forward(x_t, h_prev, W_hh, W_xh, b_h):
    """
    RNN单元的前向传播
    
    Args:
        x_t: 输入，形状 (batch_size, input_size)
        h_prev: 上一隐藏状态，形状 (batch_size, hidden_size)
        W_hh: 隐藏到隐藏权重，形状 (hidden_size, hidden_size)
        W_xh: 输入到隐藏权重，形状 (hidden_size, input_size)
        b_h: 偏置，形状 (hidden_size,)
    
    Returns:
        h_t: 当前隐藏状态，形状 (batch_size, hidden_size)
        cache: 缓存用于反向传播
    """
    # 计算当前隐藏状态
    h_t = np.tanh(np.dot(x_t, W_xh.T) + np.dot(h_prev, W_hh.T) + b_h)
    
    # 缓存用于反向传播
    cache = (x_t, h_prev, h_t, W_hh, W_xh, b_h)
    
    return h_t, cache

def rnn_cell_backward(dh_next, cache):
    """
    RNN单元的单步反向传播
    
    Args:
        dh_next: 损失对h_t的梯度，形状 (batch_size, hidden_size)
        cache: 前向传播缓存
    
    Returns:
        dx_t: 损失对x_t的梯度
        dh_prev: 损失对h_prev的梯度
        dW_hh: 损失对W_hh的梯度
        dW_xh: 损失对W_xh的梯度
        db_h: 损失对b_h的梯度
    """
    x_t, h_prev, h_t, W_hh, W_xh, b_h = cache
    
    # tanh的导数: 1 - tanh^2
    dtanh = dh_next * (1 - h_t**2)
    
    # 计算各梯度
    batch_size = x_t.shape[0]
    
    # dW_xh = dtanh^T @ x_t
    dW_xh = np.dot(dtanh.T, x_t)
    
    # dW_hh = dtanh^T @ h_prev
    dW_hh = np.dot(dtanh.T, h_prev)
    
    # db_h = sum(dtanh, axis=0)
    db_h = np.sum(dtanh, axis=0)
    
    # dx_t = dtanh @ W_xh
    dx_t = np.dot(dtanh, W_xh)
    
    # dh_prev = dtanh @ W_hh
    dh_prev = np.dot(dtanh, W_hh)
    
    return dx_t, dh_prev, dW_hh, dW_xh, db_h

# 测试
print("\n" + "="*50)
print("3.2 RNN单元测试:")
batch_size, input_size, hidden_size = 3, 4, 5
x_t = np.random.randn(batch_size, input_size)
h_prev = np.random.randn(batch_size, hidden_size)
W_hh = np.random.randn(hidden_size, hidden_size)
W_xh = np.random.randn(hidden_size, input_size)
b_h = np.random.randn(hidden_size)

h_t, cache = rnn_cell_forward(x_t, h_prev, W_hh, W_xh, b_h)
dh_next = np.random.randn(batch_size, hidden_size)

dx_t, dh_prev, dW_hh, dW_xh, db_h = rnn_cell_backward(dh_next, cache)

print(f"h_t形状: {h_t.shape}")
print(f"dx_t形状: {dx_t.shape}")
print(f"dh_prev形状: {dh_prev.shape}")
print(f"dW_hh形状: {dW_hh.shape}")
print(f"dW_xh形状: {dW_xh.shape}")
print(f"db_h形状: {db_h.shape}")



3.2 RNN单元测试:
h_t形状: (3, 5)
dx_t形状: (3, 4)
dh_prev形状: (3, 5)
dW_hh形状: (5, 5)
dW_xh形状: (5, 4)
db_h形状: (5,)


In [12]:
# 4.1 理论计算题 - 参数计算验证
# ============================================================================

def calculate_bidirectional_rnn_params(L, H, D, O):
    """
    计算深度双向RNN的参数总数
    
    Args:
        L: 层数
        H: 每层隐藏单元数
        D: 输入维度
        O: 输出维度
    
    Returns:
        total_params: 参数总数
    """
    # 第1层：输入维度为D
    # 每层有前向和后向两个方向
    # 每个方向：H*(D+H) 权重 + H*H 隐藏权重 + H 偏置 = H*(D+H) + H*H + H = H*(D+2H+1)
    layer1_params = 2 * H * (D + H + 1)
    
    # 第l层（l>1）：输入维度为2H（前一层双向拼接）
    # 每个方向：H*(2H+H) + H*H + H = H*(3H+1)
    other_layers_params = 2 * H * (3 * H + 1) * (L - 1)
    
    # 输出层参数（如果有输出层）
    output_params = O * (2 * H + 1) if O > 0 else 0
    
    total = layer1_params + other_layers_params + output_params
    
    return {
        'layer1': layer1_params,
        'other_layers': other_layers_params,
        'output': output_params,
        'total': total
    }

print("\n" + "="*50)
print("4.1 深度双向RNN参数计算:")

# 测试不同配置
configs = [
    (1, 128, 100, 50),
    (2, 128, 100, 50),
    (3, 256, 100, 50),
    (2, 256, 512, 100),
]

for L, H, D, O in configs:
    params = calculate_bidirectional_rnn_params(L, H, D, O)
    print(f"\nL={L}, H={H}, D={D}, O={O}:")
    print(f"  第1层参数: {params['layer1']:,}")
    print(f"  其他层参数: {params['other_layers']:,}")
    print(f"  输出层参数: {params['output']:,}")
    print(f"  总参数: {params['total']:,}")


4.1 深度双向RNN参数计算:

L=1, H=128, D=100, O=50:
  第1层参数: 58,624
  其他层参数: 0
  输出层参数: 12,850
  总参数: 71,474

L=2, H=128, D=100, O=50:
  第1层参数: 58,624
  其他层参数: 98,560
  输出层参数: 12,850
  总参数: 170,034

L=3, H=256, D=100, O=50:
  第1层参数: 182,784
  其他层参数: 787,456
  输出层参数: 25,650
  总参数: 995,890

L=2, H=256, D=512, O=100:
  第1层参数: 393,728
  其他层参数: 393,728
  输出层参数: 51,300
  总参数: 838,756


In [13]:
# 4.2 编程题
# ============================================================================

import torch
import torch.nn as nn

class BidirectionalRNNEncoder(nn.Module):
    """
    双向RNN编码器（使用PyTorch的RNN）
    """
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super(BidirectionalRNNEncoder, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=False
        )
    
    def forward(self, X):
        outputs, h_n = self.rnn(X)
        
        # 取最后一层的前向和后向
        forward_final = h_n[-2, :, :]
        backward_final = h_n[-1, :, :]
        final_state = torch.cat([forward_final, backward_final], dim=1)
        
        return outputs, final_state

# 手动实现的双向RNN
class BidirectionalRNNEncoderManual(nn.Module):
    """
    手动实现的双向RNN编码器
    """
    def __init__(self, input_dim, hidden_dim):
        super(BidirectionalRNNEncoderManual, self).__init__()
        self.hidden_dim = hidden_dim
        
        self.W_xh_f = nn.Parameter(torch.randn(hidden_dim, input_dim) * 0.01)
        self.W_hh_f = nn.Parameter(torch.randn(hidden_dim, hidden_dim) * 0.01)
        self.b_h_f = nn.Parameter(torch.zeros(hidden_dim))
        
        self.W_xh_b = nn.Parameter(torch.randn(hidden_dim, input_dim) * 0.01)
        self.W_hh_b = nn.Parameter(torch.randn(hidden_dim, hidden_dim) * 0.01)
        self.b_h_b = nn.Parameter(torch.zeros(hidden_dim))
    
    def forward(self, X):
        seq_len, batch_size, _ = X.shape
        
        # 前向传播
        h_forward = []
        h_t = torch.zeros(batch_size, self.hidden_dim)
        for t in range(seq_len):
            x_t = X[t]
            h_t = torch.tanh(torch.mm(x_t, self.W_xh_f.T) + torch.mm(h_t, self.W_hh_f.T) + self.b_h_f)
            h_forward.append(h_t)
        
        # 后向传播
        h_backward = []
        h_t = torch.zeros(batch_size, self.hidden_dim)
        for t in range(seq_len - 1, -1, -1):
            x_t = X[t]
            h_t = torch.tanh(torch.mm(x_t, self.W_xh_b.T) + torch.mm(h_t, self.W_hh_b.T) + self.b_h_b)
            h_backward.insert(0, h_t)
        
        # 拼接
        outputs = torch.stack([torch.cat([h_forward[t], h_backward[t]], dim=1) for t in range(seq_len)])
        final_state = outputs[-1]
        
        return outputs, final_state

# 测试
print("\n" + "="*50)
print("4.2 双向RNN编码器测试:")
seq_len, batch, input_dim, hidden_dim = 5, 3, 8, 10
X = torch.randn(seq_len, batch, input_dim)

encoder = BidirectionalRNNEncoder(input_dim, hidden_dim)
outputs, final_state = encoder(X)

print("PyTorch实现:")
print(f"  outputs形状: {outputs.shape}")
print(f"  final_state形状: {final_state.shape}")

encoder_manual = BidirectionalRNNEncoderManual(input_dim, hidden_dim)
outputs_manual, final_state_manual = encoder_manual(X)

print("\n手动实现:")
print(f"  outputs形状: {outputs_manual.shape}")
print(f"  final_state形状: {final_state_manual.shape}")



4.2 双向RNN编码器测试:
PyTorch实现:
  outputs形状: torch.Size([5, 3, 20])
  final_state形状: torch.Size([3, 20])

手动实现:
  outputs形状: torch.Size([5, 3, 20])
  final_state形状: torch.Size([3, 20])


In [14]:
# 5.1 理论计算题 - 负采样演示
# ============================================================================

def demonstrate_negative_sampling():
    """
    演示Skip-gram负采样的损失函数计算
    """
    import numpy as np
    
    # 模拟词频分布
    vocab_size = 100
    word_freq = np.random.randint(1, 1000, vocab_size)
    word_freq = word_freq / word_freq.sum()
    
    # 计算3/4次方分布
    word_freq_pow = word_freq ** 0.75
    noise_dist = word_freq_pow / word_freq_pow.sum()
    
    # 模拟词向量
    d = 50
    v_c = np.random.randn(d)
    u_o = np.random.randn(d)
    K = 5
    
    # 采样负样本
    negative_samples = np.random.choice(vocab_size, K, p=noise_dist)
    u_neg = np.random.randn(K, d)
    
    # 计算损失
    def sigmoid(x):
        return 1 / (1 + np.exp(-x))
    
    # 正样本损失
    pos_loss = -np.log(sigmoid(np.dot(v_c, u_o)))
    
    # 负样本损失
    neg_loss = 0
    for k in range(K):
        neg_loss += -np.log(sigmoid(-np.dot(v_c, u_neg[k])))
    
    total_loss = pos_loss + neg_loss
    
    print("\n" + "="*50)
    print("5.1 Skip-gram负采样演示:")
    print(f"词汇表大小: {vocab_size}")
    print(f"负样本数量 K: {K}")
    print(f"正样本损失: {pos_loss:.4f}")
    print(f"负样本损失: {neg_loss:.4f}")
    print(f"总损失: {total_loss:.4f}")
    print(f"噪声分布前5个词概率: {noise_dist[:5]}")

demonstrate_negative_sampling()


5.1 Skip-gram负采样演示:
词汇表大小: 100
负样本数量 K: 5
正样本损失: 0.0004
负样本损失: 27.2143
总损失: 27.2147
噪声分布前5个词概率: [0.00437307 0.00933448 0.01791762 0.00728142 0.01247627]


In [15]:
# 5.2 编程题
# ============================================================================

import torch
import torch.nn.functional as F

def cbow_forward(context_indices, target_idx, W, W_out):
    """
    CBOW模型的前向传播和损失计算（完整softmax）
    
    Args:
        context_indices: 上下文词索引列表，形状 (batch_size, context_size)
        target_idx: 目标中心词索引，形状 (batch_size,)
        W: 输入权重矩阵，形状 (V, d)
        W_out: 输出权重矩阵，形状 (d, V)
    
    Returns:
        loss: 交叉熵损失
        hidden: 隐藏层（平均上下文向量）
        probs: 输出概率分布
    """
    batch_size, context_size = context_indices.shape
    V, d = W.shape
    
    # 获取上下文词的嵌入向量
    context_embeddings = W[context_indices]
    
    # 计算平均上下文向量作为隐藏层
    hidden = torch.mean(context_embeddings, dim=1)
    
    # 计算输出得分
    scores = torch.mm(hidden, W_out)
    
    # 应用softmax
    probs = F.softmax(scores, dim=1)
    
    # 计算交叉熵损失
    loss = F.cross_entropy(scores, target_idx)
    
    return loss, hidden, probs

print("\n" + "="*50)
print("5.2 CBOW模型测试:")
V, d, batch_size, context_size = 100, 50, 4, 3
W = torch.randn(V, d, requires_grad=True)
W_out = torch.randn(d, V, requires_grad=True)

context_indices = torch.randint(0, V, (batch_size, context_size))
target_idx = torch.randint(0, V, (batch_size,))

loss, hidden, probs = cbow_forward(context_indices, target_idx, W, W_out)

print(f"损失值: {loss.item():.4f}")
print(f"隐藏层形状: {hidden.shape}")
print(f"概率分布形状: {probs.shape}")
print(f"每行概率之和: {probs.sum(dim=1)}")


5.2 CBOW模型测试:
损失值: 11.8687
隐藏层形状: torch.Size([4, 50])
概率分布形状: torch.Size([4, 100])
每行概率之和: tensor([1.0000, 1.0000, 1.0000, 1.0000], grad_fn=<SumBackward1>)


In [16]:
# 6.1 理论计算题 - 代码验证
# ============================================================================

import numpy as np

def softmax(x, axis=1):
    """Softmax函数"""
    exp_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

print("\n" + "="*50)
print("6.1 缩放点积注意力计算:")

# 设置随机种子以便复现
np.random.seed(42)

Q = np.random.randn(2, 4)
K = np.random.randn(3, 4)
V = np.random.randn(3, 5)
d_k = 4

print(f"Q形状: {Q.shape}")
print(f"K形状: {K.shape}")
print(f"V形状: {V.shape}")

# 1. 计算得分矩阵
scores = np.dot(Q, K.T) / np.sqrt(d_k)
print(f"\n得分矩阵 S (形状 {scores.shape}):")
print(np.round(scores, 3))

# 2. Softmax
attention_weights = softmax(scores)
print(f"\n注意力权重 α (每行之和为1):")
print(np.round(attention_weights, 3))
print(f"行和: {np.round(attention_weights.sum(axis=1), 3)}")

# 3. 加权求和
output = np.dot(attention_weights, V)
print(f"\n输出矩阵 O (形状 {output.shape}):")
print(np.round(output, 3))


6.1 缩放点积注意力计算:
Q形状: (2, 4)
K形状: (3, 4)
V形状: (3, 5)

得分矩阵 S (形状 (2, 3)):
[[-0.659 -0.794 -1.643]
 [-0.553 -1.382 -1.177]]

注意力权重 α (每行之和为1):
[[0.445 0.389 0.166]
 [0.507 0.221 0.272]]
行和: [1. 1.]

输出矩阵 O (形状 (2, 5)):
[[ 0.595 -0.24   0.174 -1.043 -0.219]
 [ 0.604  0.134  0.114 -1.143 -0.117]]


In [17]:
# 6.2 编程题
# ============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    """
    多头注意力机制
    """
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # 线性投影层
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
    
    def forward(self, X):
        """
        Args:
            X: 输入，形状 (seq_len, batch, d_model)
        
        Returns:
            output: 输出，形状 (seq_len, batch, d_model)
        """
        seq_len, batch_size, _ = X.shape
        
        # 线性投影
        Q = self.W_q(X)
        K = self.W_k(X)
        V = self.W_v(X)
        
        # 重塑为多头形式
        Q = Q.view(seq_len, batch_size, self.num_heads, self.d_k).transpose(0, 1).transpose(1, 2)
        Q = Q.contiguous().view(batch_size * self.num_heads, seq_len, self.d_k)
        
        K = K.view(seq_len, batch_size, self.num_heads, self.d_k).transpose(0, 1).transpose(1, 2)
        K = K.contiguous().view(batch_size * self.num_heads, seq_len, self.d_k)
        
        V = V.view(seq_len, batch_size, self.num_heads, self.d_k).transpose(0, 1).transpose(1, 2)
        V = V.contiguous().view(batch_size * self.num_heads, seq_len, self.d_k)
        
        # 缩放点积注意力
        scores = torch.bmm(Q, K.transpose(1, 2)) / (self.d_k ** 0.5)
        attention_weights = F.softmax(scores, dim=-1)
        
        # 加权求和
        head_outputs = torch.bmm(attention_weights, V)
        
        # 重塑回原始形状
        head_outputs = head_outputs.view(batch_size, self.num_heads, seq_len, self.d_k)
        head_outputs = head_outputs.transpose(0, 1).transpose(1, 2)
        head_outputs = head_outputs.contiguous().view(seq_len, batch_size, self.d_model)
        
        # 最终线性层
        output = self.W_o(head_outputs)
        
        return output

print("\n" + "="*50)
print("6.2 多头注意力测试:")
seq_len, batch, d_model, num_heads = 8, 4, 4, 2
X = torch.randn(seq_len, batch, d_model)

mha = MultiHeadAttention(d_model, num_heads)
output = mha(X)

print(f"输入形状: {X.shape}")
print(f"输出形状: {output.shape}")
print(f"输入和输出形状相同: {X.shape == output.shape}")
print(f"d_model = {d_model}, num_heads = {num_heads}")
print(f"每个头的维度 d_k = d_v = {d_model // num_heads}")


6.2 多头注意力测试:
输入形状: torch.Size([8, 4, 4])
输出形状: torch.Size([8, 4, 4])
输入和输出形状相同: True
d_model = 4, num_heads = 2
每个头的维度 d_k = d_v = 2
